# 01 — Data Pipeline: Generate Contrastive Triplets

This notebook generates the training data for the HCCS scorer.

**What it produces:** `data/triplets.jsonl` — ~1,500–3,000 contrastive triplets  
of the form `(query, positive_context, negative_context, hallucination_type)`.

**How it works:**
1. Load tasks from CodeHaluEval (HuggingFace: `yuchen814/CodeHalu`)
2. For each task, sample 6 random context subsets from the repo chunks
3. Generate code with each subset using DeepSeek-Coder-1.3B
4. Execute each generated code against the task's test cases
5. Pair a PASSING subset with a FAILING subset → contrastive triplet
6. Save triplets incrementally to Drive (Colab-safe)

**Runtime:** ~10 hours on T4 GPU (~2 CU)

---
All cells are **idempotent** — safe to re-run after disconnect.

## Setup

In [ ]:
!pip install torch transformers datasets rank-bm25 tqdm --quiet

In [ ]:
import sys, os
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/HaluGuard'
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -e '.[dev]' -q

from notebooks.utils import check_gpu, print_env_summary, get_drive_path, append_jsonl, count_jsonl
DEVICE = check_gpu()
print_env_summary()

In [ ]:
from pathlib import Path
from tqdm import tqdm

from haluguard.data_pipeline import (
    ContrastiveTriplet,
    create_triplets_from_task,
    load_triplets,
    save_triplets,
    summarise_triplets,
)

DRIVE_DIR    = get_drive_path('HaluGuard/data')
TRIPLETS_PATH = DRIVE_DIR / 'triplets.jsonl'
print(f'Triplets path: {TRIPLETS_PATH}')
print(f'Existing triplets: {count_jsonl(TRIPLETS_PATH)}')

## 1. Load and Inspect Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('yuchen814/CodeHalu', split='train')
print(f'Dataset size: {len(ds)} samples')
print(f'Columns: {ds.column_names}')

In [ ]:
# IMPORTANT: Run this cell first to see the actual column names.
# The field names below (query, repo_files, test_code, task_id) are placeholders
# — update FIELD_MAP in the next cell to match what you see here.
sample = ds[0]
for key, val in sample.items():
    preview = str(val)[:150] + '...' if len(str(val)) > 150 else str(val)
    print(f'{key!r:25s}: {preview}')

In [ ]:
# --- UPDATE THESE after inspecting the sample above ---
# Map from our internal names → actual dataset column names
FIELD_MAP = {
    'task_id':    'task_id',    # unique task identifier
    'query':      'prompt',     # natural-language task description
    'repo_files': 'context',    # dict of {filepath: source} OR a string
    'test_code':  'test',       # test assertions string
}

def extract_task(sample: dict) -> dict:
    """Extract fields from a dataset sample using FIELD_MAP.
    
    Handles the case where repo_files is a JSON string rather than a dict.
    """
    import json
    repo_files = sample[FIELD_MAP['repo_files']]
    if isinstance(repo_files, str):
        try:
            repo_files = json.loads(repo_files)
        except json.JSONDecodeError:
            # Treat as a single file if it's plain source code
            repo_files = {'main.py': repo_files}
    return {
        'task_id':    str(sample[FIELD_MAP['task_id']]),
        'query':      sample[FIELD_MAP['query']],
        'repo_files': repo_files,
        'test_code':  sample[FIELD_MAP['test_code']],
    }

# Smoke test
task = extract_task(ds[0])
print('Extracted task keys:', list(task.keys()))
print('repo_files type:', type(task['repo_files']))
print('test_code preview:', task['test_code'][:100])

## 2. Load Code Generation Model (DeepSeek-Coder 1.3B)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-instruct'

print(f'Loading {MODEL_NAME}...')
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
gen_model.eval()
print(f'Model loaded on {next(gen_model.parameters()).device}')
print(f'Model size: {sum(p.numel() for p in gen_model.parameters()) / 1e6:.0f}M params')

In [ ]:
def generate_code(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate Python code from a prompt using DeepSeek-Coder-1.3B.
    
    Temperature 0.6: low enough for coherent code, high enough for
    diverse outputs across different context subsets (essential for
    getting both PASS and FAIL outcomes to form triplets).
    """
    inputs = gen_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    # Strip the prompt tokens — only decode the newly generated part
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True)


# Quick smoke test (1 generation)
test_prompt = '# Task: return the sum of two numbers\n\ndef add(a, b):\n'
output = generate_code(test_prompt, max_new_tokens=50)
print('Smoke test output:', repr(output[:100]))

## 3. Generate Contrastive Triplets

In [ ]:
import json
from haluguard.data_pipeline import _triplet_to_dict

N_TASKS    = 500   # tasks to process (start with 10 to validate, then scale to 500)
N_SUBSETS  = 6    # context subsets per task
SEED       = 42

already_done_ids = set()
if TRIPLETS_PATH.exists():
    with TRIPLETS_PATH.open() as f:
        for line in f:
            if line.strip():
                already_done_ids.add(json.loads(line).get('task_id', ''))
print(f'Already processed {len(already_done_ids)} tasks, {count_jsonl(TRIPLETS_PATH)} triplets')

new_triplets = 0
tasks_to_run = [s for s in ds.select(range(N_TASKS)) if str(s[FIELD_MAP['task_id']]) not in already_done_ids]
print(f'Tasks to process: {len(tasks_to_run)}')

for i, sample in enumerate(tqdm(tasks_to_run, desc='Generating triplets')):
    task = extract_task(sample)
    try:
        triplets = create_triplets_from_task(
            task_id=task['task_id'],
            query=task['query'],
            repo_files=task['repo_files'],
            test_code=task['test_code'],
            generate_fn=generate_code,
            n_subsets=N_SUBSETS,
            seed=SEED + i,
        )
        for triplet in triplets:
            append_jsonl(_triplet_to_dict(triplet), TRIPLETS_PATH)
            new_triplets += 1
    except Exception as e:
        # Log and continue — don't let one bad task kill the run
        print(f'  Task {task["task_id"]} failed: {e}')
        continue

print(f'Done. Added {new_triplets} new triplets. Total: {count_jsonl(TRIPLETS_PATH)}')

## 4. Analyse Triplets

In [ ]:
if TRIPLETS_PATH.exists():
    triplets = load_triplets(TRIPLETS_PATH)
    summary  = summarise_triplets(triplets)
    print(f'Total triplets: {summary["total"]}')
    print('By hallucination type:')
    for t, count in summary['by_hallucination_type'].items():
        pct = 100 * count / max(summary['total'], 1)
        print(f'  {t:10s}: {count:5d} ({pct:.1f}%)')
    print(f'Unknown type: {summary["unknown"]}')

    if summary['total'] < 100:
        print('\nWARNING: fewer than 100 triplets — increase N_TASKS or check that tasks produce failures.')
    elif summary['total'] >= 1000:
        print('\nReady for training! Run notebook 02 next.')
else:
    print('No triplets yet — run the generation cell above first.')

In [ ]:
# Save progress to GitHub
import subprocess
os.chdir(REPO_DIR)
!git add -A
!git commit -m 'chore: update after notebook 01 data generation'
!git push